# Profiler Summary

Notebook para leer corridas en `profiler/logs/<EXPERIMENT_NAME>`, recalcular métricas desde `trace` y exportar CSVs de resumen.


In [7]:
from pathlib import Path
import sys
from IPython.display import display

try:
    from profiler.profiler_tools import export_core_summary, make_run_logger, get_timestamp
except ModuleNotFoundError:
    candidates = [Path.cwd(), Path.cwd().parent]
    for cand in candidates:
        if (cand / "profiler" / "profiler_tools.py").exists() and str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
    from profiler.profiler_tools import export_core_summary, make_run_logger, get_timestamp


def find_project_root():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / "src").exists() and (p / "profiler").exists():
            return p
    raise RuntimeError("No se encontro project root (esperado: carpetas src/ y profiler/).")


PROJECT_ROOT = find_project_root()
PROFILER_DIR = PROJECT_ROOT / "profiler"
EXPERIMENT_NAME = "diffusion_profiler_light_batch"
OUTPUT_TAG = "batch"
LOG_ROOT = PROFILER_DIR / "logs" / EXPERIMENT_NAME
CORE_CSV = PROFILER_DIR / f"profiler_runs_summary_core_{OUTPUT_TAG}.csv"
FULL_CSV = PROFILER_DIR / f"profiler_runs_summary_{OUTPUT_TAG}.csv"

summary_job_dir = PROFILER_DIR / "logs" / "summary_jobs" / f"summary_{get_timestamp()}"
log_event = make_run_logger(summary_job_dir, run_name="summary_job")

print(f"LOG_ROOT={LOG_ROOT}")
print(f"CORE_CSV={CORE_CSV}")
print(f"events={summary_job_dir / 'events.jsonl'}")

LOG_ROOT=/home/gkulemeyer/Documents/Repos/RNADiffusion/profiler/logs/diffusion_profiler_light_batch
CORE_CSV=/home/gkulemeyer/Documents/Repos/RNADiffusion/profiler/profiler_runs_summary_core_batch.csv
events=/home/gkulemeyer/Documents/Repos/RNADiffusion/profiler/logs/summary_jobs/summary_20260304_011734/events.jsonl


In [8]:
log_event("run_started", {"log_root": str(LOG_ROOT), "output_core": str(CORE_CSV)})

try:
    full_df, core_df = export_core_summary(
        log_root=LOG_ROOT,
        output_csv=CORE_CSV,
        output_full_csv=FULL_CSV,
    )

    log_event(
        "summary_saved",
        {
            "rows_full": int(len(full_df)),
            "rows_core": int(len(core_df)),
            "core_csv": str(CORE_CSV),
            "full_csv": str(FULL_CSV),
        },
    )
    log_event("run_finished", {"status": "ok"})

except Exception as err:
    log_event("run_error", {"message": str(err)}, level="error")
    raise

if len(core_df) == 0:
    print("No se encontraron corridas para resumir.")
else:
    display(core_df.sort_values(["timesteps", "use_amp"]).head(30))

,run_name,status,timesteps,batch_size,grad_accum_steps,use_amp,oom_events,trace_path,trace_size_mb,train_batch_ms,...,train_backward_alloc_traffic_mb_gpu,train_backward_free_traffic_mb_gpu,train_step_alloc_traffic_mb_gpu,train_step_free_traffic_mb_gpu,train_forward_alloc_traffic_mb_cpu,train_forward_free_traffic_mb_cpu,train_backward_alloc_traffic_mb_cpu,train_backward_free_traffic_mb_cpu,train_step_alloc_traffic_mb_cpu,train_step_free_traffic_mb_cpu
0,batch_ga_sweep_sim90_bs1_ga1_pb4_10ts_fp32,ok,10,1,1,False,0.0,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,126.943350,304.657000,...,11105.359863,11729.752767,10.642904,10.642904,0.001160,0.001152,0.000153,0.000160,0.000938,0.000938
1,batch_ga_sweep_sim90_bs1_ga2_pb4_10ts_fp32,ok,10,1,2,False,0.0,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,119.311906,233.342000,...,10516.226074,11084.727051,20.775391,10.387695,0.001160,0.001152,0.000153,0.000160,0.001095,0.000938
2,batch_ga_sweep_sim90_bs1_ga8_pb4_10ts_fp32,ok,10,1,8,False,0.0,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,121.379138,300.359667,...,9632.360840,10206.355306,31.163086,10.387695,0.001160,0.001152,0.000153,0.000160,0.001251,0.000938
3,batch_ga_sweep_sim90_bs4_ga1_pb4_10ts_fp32,ok,10,4,1,False,0.0,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,119.977223,245.755000,...,12113.355957,15095.801270,10.481445,10.481445,0.001160,0.001152,0.000153,0.000160,0.000938,0.000938
4,batch_ga_sweep_sim90_bs4_ga2_pb4_10ts_fp32,ok,10,4,2,False,0.0,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,118.976442,306.180667,...,14841.738770,18456.573242,21.509766,11.122070,0.001160,0.001152,0.000153,0.000160,0.001095,0.000938
5,batch_ga_sweep_sim90_bs4_ga8_pb4_10ts_fp32,ok,10,4,8,False,0.0,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,115.108774,226.641333,...,17735.467773,21515.707194,32.731445,11.122070,0.001160,0.001152,0.000153,0.000160,0.001251,0.000938
6,batch_ga_sweep_sim90_bs1_ga1_pb4_10ts_amp,ok,10,1,1,True,0.0,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,150.929397,402.212333,...,6376.860840,6994.212891,11.010417,11.010417,0.001160,0.001152,0.000153,0.000160,0.000938,0.000938
7,batch_ga_sweep_sim90_bs1_ga2_pb4_10ts_amp,ok,10,1,2,True,0.0,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,148.617901,420.739333,...,8106.294108,8619.881673,21.400391,10.798828,0.001160,0.001152,0.000153,0.000160,0.001095,0.000938
8,batch_ga_sweep_sim90_bs1_ga8_pb4_10ts_amp,ok,10,1,8,True,0.0,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,140.530616,241.767333,...,6479.643392,6880.775553,31.751953,10.390625,0.001160,0.001152,0.000153,0.000160,0.001251,0.000938
9,batch_ga_sweep_sim90_bs4_ga1_pb4_10ts_amp,ok,10,4,1,True,0.0,/home/gkulemeyer/Documents/Repos/RNADiffusion/...,146.540525,391.383333,...,15394.493164,18718.732585,18.588216,11.243164,0.001160,0.001152,0.000153,0.000160,0.001043,0.000938
